In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# 1. Datan lataaminen
df = pd.read_csv('cicids2017_cleaned.csv')

# Käsitellään äärettömyydet ja puuttuvat arvot
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Valitaan 10 parasta ominaisuutta
selected_features = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets',
    'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Flow Bytes/s', 'Flow Packets/s',
    'Fwd Header Length', 'Bwd Header Length'
]
label = 'Attack Type'

# Otetaan 20 000 rivin otos ja siivotaan harvinaiset luokat (väh. 6 näytettä)
# SVM mallin kouluttaminen 20 000 rivin aineistolla kesti 4 minuuttia ja 21 sekuntia.
# Ajanpuutteen vuoksi rivejä ei lisätty, vaikka se todennäköisesti parantaisi mallin suorituskykyä.
df = df[selected_features + [label]].sample(n=20000, random_state=42)
v_counts = df[label].value_counts()
df = df[df[label].isin(v_counts[v_counts >= 6].index)]

X = df.drop(label, axis=1)
y = df[label]

# 2. Datan jako (60/20/20)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# 3. Mallien määrittely
# Määritellään vain muuttuvat osat: algoritmi ja sen omat parametrit
models = {
    'SVM': (SVC(cache_size=2000), {
        'pca__n_components': [8],
        'model__C': [1, 10],
        'model__kernel': ['rbf'],
        'smote__k_neighbors': [1, 3]
    }),
    'Naive Bayes': (GaussianNB(), {
        'pca__n_components': [5, 8],
        'model__var_smoothing': [1e-9, 1e-8],
        'smote__k_neighbors': [1, 3]
    })
}

# 4. Mallien harjoittaminen ja tulostus silmukassa
for name, (algorithm, params) in models.items():
    print(f"\n{'='*10} Aloitetaan {name} Grid Search {'='*10}")

    # Rakennetaan putki (yhteiset osat + vaihtuva algoritmi)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(random_state=42)),
        ('smote', SMOTE(random_state=42)),
        ('model', algorithm)  # 'model' toimii yleisnimenä molemmille algoritmeille
    ])

    # Suoritetaan Grid Search
    grid = GridSearchCV(pipe, params, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
    grid.fit(X_train, y_train)

    # Tulosten tulostus
    y_pred = grid.predict(X_test)
    print(f"\n--- {name.upper()} TULOKSET ---")
    print(f"Parhaat parametrit: {grid.best_params_}")
    print(f"Testitarkkuus: {accuracy_score(y_test, y_pred):.4f}")

    print("\nSekaannusmatriisi:")
    print(confusion_matrix(y_test, y_pred))

    print("\nLuokitteluraportti:")
    print(classification_report(y_test, y_pred))


========== Aloitetaan SVM Grid Search ==========
Fitting 3 folds for each of 4 candidates, totalling 12 fits

--- SVM TULOKSET ---
Parhaat parametrit: {'model__C': 10, 'model__kernel': 'rbf', 'pca__n_components': 8, 'smote__k_neighbors': 3}
Testitarkkuus: 0.8958

Sekaannusmatriisi:
[[  50    0    0    0]
 [ 133 2797  128  102]
 [  13   18  721   21]
 [   1    1    0   15]]

Luokitteluraportti:
                precision    recall  f1-score   support

   Brute Force       0.25      1.00      0.40        50
Normal Traffic       0.99      0.89      0.94      3160
 Port Scanning       0.85      0.93      0.89       773
   Web Attacks       0.11      0.88      0.19        17

      accuracy                           0.90      4000
     macro avg       0.55      0.93      0.61      4000
  weighted avg       0.95      0.90      0.92      4000


========== Aloitetaan Naive Bayes Grid Search ==========
Fitting 3 folds for each of 8 candidates, totalling 24 fits

--- NAIVE BAYES TULOKSET ---
Par